In [20]:
import random
from datetime import datetime, timedelta
import pandas as pd
import pyodbc
random.seed(42)

number = 100

titles = ['Master', 'Doctor', 'Habilitated Doctor', 'University Professor', 'Professor']

salary_range = {
    'Master': (3000, 5000),
    'Doctor': (5000, 7000),
    'Habilitated Doctor': (7000, 9000),
    'University Professor': (8000, 12000),
    'Professor': (10000, 16000)
}

last_names = [
    'Baggins', 'Took', 'Brandybuck', 'Gamgee', 'Cotton',
    'Durin', 'Oakenshield', 'Ironfoot', 'Balin', 'Thorin',
    'Stormwind', 'Blackstone', 'Thundershield', 'Frostwhisper', 'Deepstone', 'Ironfist',
    'Starfire', 'Sunblaze', 'Shadowhunter', 'Ravenshadow',
    'Arren', 'Ged', 'Tormer', 'Morwen', 'Tenar', 'Ceredin', 'Duny', 'Yarrow', 'Dragonking', 'Otter',
    'Kelsier', 'Vin','Tindwyl', 'Sazed', 'Marsh', 'Spook', 'Breeze', 'Clubs',
    'Shade', 'Varden', 'Brom', 'Saphira', 'Roran', 'Murtagh', 'Arya', 'Oromis',
    'Pevensie', 'Aslan', 'Tumnus', 'Shasta', 'Reepicheep',
    'Galathor', 'Ravenshadow', 'Stormblade', 'Ironfoot', 'Sunstrike', 'Starfury', 'Dragonsworn',
    'Shadowmoon', 'Tormar', 'Ironfist', 'Blackthorne', 'Everwinter', 'Lightbringer', 'Blackwood',
    'Shadowhunter', 'Redwyne', 'Iceheart', 'Thornfield', 'Nightbloom', 'Frostfall', 'Stonehelm', 'Duskbane', 'Flameborn', 'Moonshade', 'Ashenforge',
    'Stormrend', 'Nightforge', 'Frostborn', 'Grimward', 'Wolfsbane',
    'Ironveil', 'Silverthorn', 'Darkmere', 'Brightflame', 'Thornhelm',
    'Emberlyn', 'Voidwalker', 'Windrider', 'Duskwatch', 'Blazewind'
]

conn_str = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=DESKTOP-4TCLG8I;DATABASE=university;Trusted_Connection=yes'
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

names = pd.read_csv('characters_no_surnames.csv').iloc[:, 0]

cursor.execute("SELECT DepartmentName FROM Department")
department_names = [row[0] for row in cursor.fetchall()]

def generate_random_birth_date_1():
    start_date = datetime(1940, 1, 1)
    end_date = datetime(1990, 1, 1)
    delta = end_date - start_date
    return start_date + timedelta(days=random.randint(0, delta.days))

used_name_combinations = set()
teachers = []

while len(teachers) < number:
    first_name = random.choice(names)
    last_name = random.choice(last_names)
    name_pair = (first_name, last_name)

    if name_pair in used_name_combinations:
        continue  # Try again with a new combination

    used_name_combinations.add(name_pair)
    title = random.choice(titles)
    departmentName = random.choice(department_names)
    salary = random.randint(*salary_range[title])
    birth_date = generate_random_birth_date_1()
    employment_date = birth_date.replace(year=birth_date.year + random.randint(22, 30))

    teacher = {
        'LecturerID': len(teachers) + 1,
        'FirstName': first_name,
        'LastName': last_name,
        'Title': title,
        'Email': f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 99)}@middleearth.edu",
        'DepartmentName': departmentName,
        'MonthlySalary': salary,
        'DateOfBirth': birth_date.strftime('%Y-%m-%d'),
        'DateOfEmployment': employment_date.strftime('%Y-%m-%d')
    }
    teachers.append(teacher)




In [21]:
import pyodbc

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

for teacher in teachers:
    cursor.execute("""
        INSERT INTO UniversityTeacher (FirstName, LastName, Title, Email, DepartmentName, MonthlySalary, DateOfBirth, DateOfEmployment)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, 
    teacher['FirstName'], teacher['LastName'], teacher['Title'], teacher['Email'],
    teacher['DepartmentName'], teacher['MonthlySalary'], teacher['DateOfBirth'], teacher['DateOfEmployment'])



In [22]:
conn.commit()
cursor.close()
conn.close()

In [23]:
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
from datetime import datetime

num_students = 500

first_names = names

#last_names = pd.read_csv('lotr_surnames.csv', header=None)[0].tolist()

cursor.execute("SELECT MajorID FROM Major")
major_ids = [row[0] for row in cursor.fetchall()]

def generate_random_birth_date_2():
    start_date = datetime(1980, 1, 1)
    end_date = datetime(2008, 1, 1)
    delta = end_date - start_date
    return start_date + timedelta(days=random.randint(0, delta.days))


students = []
emails = set()


start_years = [2023, 2024, 2025]
used_names = set()

for _ in range(num_students):
    # Keep generating until we find a unique name combination
    while True:
        first_name = random.choice(first_names)
        last_name = random.choice(last_names)
        full_name = (first_name, last_name)

        if full_name not in used_names:
            used_names.add(full_name)
            break

    # Generate a unique email
    email = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 99)}@students.middleearth.edu"
    while email in emails:
        email = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 99)}@students.middleearth.edu"
    emails.add(email)
    DateOfBirth = generate_random_birth_date_2()
    # Assign major and start year
    major_id = random.choice(major_ids)
    start_year = random.choice(start_years)
    start_year_date = f"{start_year}-01-01"

    # Add student to the list
    student = (first_name, last_name, email,DateOfBirth, major_id, start_year_date)
    students.append(student)

# Insert all students into the database
cursor.executemany("""
    INSERT INTO Student (FirstName, LastName, Email,DateofBirth, MajorID, StartYear)
    VALUES (?, ?, ?, ?, ?,?)
""", students)

conn.commit()
cursor.close()
conn.close()

print(f"Inserted {num_students} students")


Inserted 500 students


In [24]:
import random
import pyodbc


conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("SELECT MajorID, Duration FROM Major")
majors = cursor.fetchall()

cursor.execute("SELECT SubjectID FROM Subject")
subject_ids = [row[0] for row in cursor.fetchall()]



subject_major_data = []

for major_id, duration in majors:
    subjects_per_semester = 3  #assuming 3 subjects per semester on average
    required_subjects = duration * subjects_per_semester

    available_subjects = subject_ids.copy()
    random.shuffle(available_subjects)

    if len(available_subjects) < required_subjects:
        print(f"Warning: Not enough available subjects for MajorID {major_id}. Available: {len(available_subjects)}, Required: {required_subjects}")
        selected_subjects = available_subjects  
    else:
        selected_subjects = available_subjects[:required_subjects] 

    for semester in range(1, duration + 1):
        start_index = (semester - 1) * subjects_per_semester
        end_index = semester * subjects_per_semester
        semester_subjects = selected_subjects[start_index:end_index]

        for subject_id in semester_subjects:
            subject_major_data.append((major_id, subject_id))

conn.commit()
print(f"Inserted {len(subject_major_data)} subject-major associations.")

cursor.close()
conn.close()


Inserted 606 subject-major associations.


In [25]:
import random
import pyodbc


conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("SELECT LecturerID FROM UniversityTeacher")
teacher_ids = [row[0] for row in cursor.fetchall()]

years = [2023, 2024, 2025]  

classes = []
subject_semester_map = {}  # assigns subject to a semester

for index, (major_id, subject_id) in enumerate(subject_major_data):
    semester = 1 if index % 2 == 0 else 2  # even indices-> semester 1, odd indices -> semester 2
    subject_semester_map[subject_id] = semester


for year_of_conduction in years:
    for major_id, subject_id in subject_major_data:
        teacher_id = random.choice(teacher_ids)  
        semester = subject_semester_map[subject_id] 
        method_of_conducting = random.choice(['stationary', 'online', 'hybrid'])

        classes.append((year_of_conduction, subject_id, major_id, semester, teacher_id, method_of_conducting))

# Wstawienie danych do bazy
cursor.executemany("""
    INSERT INTO SubjectInCertainSemester (YearOfConduction, SubjectID, MajorID, Semester, TeacherID, MethodOfConducting)
    VALUES (?, ?, ?, ?, ?, ?)
""", classes)

conn.commit()
cursor.close()
conn.close()

print(f"Inserted {len(classes)} classes into SubjectInCertainSemester table.")
display(classes)

Inserted 1818 classes into SubjectInCertainSemester table.


[(2023, 40, 1, 2, 24, 'online'),
 (2023, 13, 1, 2, 31, 'hybrid'),
 (2023, 34, 1, 2, 77, 'stationary'),
 (2023, 30, 1, 2, 37, 'hybrid'),
 (2023, 35, 1, 2, 67, 'stationary'),
 (2023, 2, 1, 1, 78, 'stationary'),
 (2023, 15, 1, 1, 31, 'hybrid'),
 (2023, 41, 1, 1, 51, 'hybrid'),
 (2023, 27, 1, 1, 40, 'online'),
 (2023, 4, 1, 1, 52, 'hybrid'),
 (2023, 42, 1, 1, 97, 'online'),
 (2023, 11, 1, 1, 10, 'stationary'),
 (2023, 5, 1, 1, 43, 'hybrid'),
 (2023, 16, 1, 2, 37, 'online'),
 (2023, 12, 1, 1, 99, 'stationary'),
 (2023, 19, 1, 1, 47, 'online'),
 (2023, 26, 1, 1, 68, 'stationary'),
 (2023, 17, 1, 1, 8, 'hybrid'),
 (2023, 26, 2, 1, 72, 'stationary'),
 (2023, 4, 2, 1, 57, 'hybrid'),
 (2023, 15, 2, 1, 79, 'hybrid'),
 (2023, 6, 2, 2, 93, 'hybrid'),
 (2023, 20, 2, 2, 23, 'stationary'),
 (2023, 8, 2, 1, 7, 'online'),
 (2023, 28, 2, 2, 59, 'hybrid'),
 (2023, 34, 2, 2, 24, 'hybrid'),
 (2023, 36, 2, 2, 19, 'stationary'),
 (2023, 32, 2, 2, 73, 'hybrid'),
 (2023, 16, 2, 2, 85, 'stationary'),
 (2023, 41,

In [26]:
import random
import pyodbc

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("""
    SELECT StudentID, StartYear, MajorID FROM Student
""")
students = cursor.fetchall()

cursor.execute("""
    SELECT ClassesID, SubjectID, MajorID, YearOfConduction, Semester FROM SubjectInCertainSemester
    WHERE YearOfConduction IN (2023, 2024, 2025)
""")
classes = cursor.fetchall()

cursor.execute("""
    SELECT MajorID, Duration FROM Major
""")
major_durations = dict(cursor.fetchall())

if not students or not classes:
    print("Brak wymaganych danych w tabelach Student lub SubjectInCertainSemester.")
    conn.close()
    exit()

major_classes = {}
for class_id, subject_id,major_id, year, semester in classes:
    if major_id not in major_classes:
        major_classes[major_id] = {}
    if year not in major_classes[major_id]:
        major_classes[major_id][year] = {}
    major_classes[major_id][year][semester] = major_classes[major_id][year].get(semester, [])
    major_classes[major_id][year][semester].append((class_id, subject_id))

grades_data = []
existing_grades = set()

for student_id, start_year, major_id in students:
    start_year = start_year.year

    duration = major_durations.get(major_id, 9)  # 9 by default, because its max duration

    semesters_completed = 0

    for year in range(start_year, 2026):
        if semesters_completed >= duration:
            break  

        if year in major_classes[major_id]: 
            for semester, subject_classes in major_classes[major_id][year].items():
                for class_id, subject_id in subject_classes:
                    if year == start_year and semester > (start_year - 2024) * 2:
                        if (student_id, class_id) not in existing_grades:
                            grade = random.choice([2, 3, 3.5, 4, 4.5, 5])

                            if grade >= 3:
                                pass_status = 1  
                                retake_status = 0  
                            else:
                                pass_status = 0  
                                retake_status = 1 

                            grades_data.append((student_id, class_id, grade, pass_status, retake_status))

                            existing_grades.add((student_id, class_id))

                            semesters_completed += 1  # increase semesters counter

unique_grades_data = list(set(grades_data))


conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

for grade_data in unique_grades_data:
    student_id, class_id, grade, pass_status, retake_status = grade_data

    cursor.execute("""
        SELECT COUNT(*) FROM Grades
        WHERE StudentID = ? AND ClassesID = ?
    """, student_id, class_id)
    
    if cursor.fetchone()[0] == 0:
        cursor.execute("""
            INSERT INTO Grades (StudentID, ClassesID, Grade, Pass, Retaking)
            VALUES (?, ?, ?, ?, ?)
        """, grade_data)


conn.commit()
print(f"Inserted {len(unique_grades_data)} grades to table Grades.")

conn.close()


Inserted 7536 grades to table Grades.


In [27]:
import random
import pandas as pd
import pyodbc

# Database connection
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Fetch all Classes data (SubjectID, Semester, Year, Teacher)
cursor.execute("""
    SELECT ClassesID, SubjectID, Semester, YearOfConduction, TeacherID
    FROM SubjectInCertainSemester
""")
classes_data = cursor.fetchall()
class_info = {row[0]: (row[1], row[2], row[3], row[4]) for row in classes_data}

# Fetch all grades
cursor.execute("""
    SELECT StudentID, ClassesID
    FROM Grades
""")
grades = cursor.fetchall()

# Survey options
most_useful_component_options = ["Lectures", "Labs", "Readings", "Assignments"]
workload_perception_options = ["Light", "Moderate", "Heavy"]
yes_no_options = ["Yes", "No"]

# Function to generate scores
def generate_score(mean, stddev, trend):
    score = random.gauss(mean, stddev) - trend
    return max(1, min(10, round(score)))

# Start building survey data
survey_data = []
survey_id = 1
mean = 8
stddev = 2

for student_id, class_id in grades:
    # Find matching class info
    if class_id not in class_info:
        continue  # skip if no matching class info found

    subject_id, semester, year, lecturer_id = class_info[class_id]

    trend = random.randint(0, 2) if year == 2025 else 0

    survey_entry = [
        survey_id,
        student_id,
        lecturer_id,
        class_id,
        subject_id,
        semester,
        year,
        max(random.randint(1, 10) - trend, 1),  # Overall Satisfaction
        max(random.randint(1, 10) - trend, 1),  # Usefulness
        random.choice(yes_no_options),          # Recommend Course
        max(random.randint(1, 10) - trend, 1),  # Clarity
        max(random.randint(1, 10) - trend, 1),  # Fairness
        max(random.randint(1, 10) - trend, 1),  # Use of Resources
        random.choice(most_useful_component_options), # Most Useful Component
        random.choice(workload_perception_options)    # Workload Perception
    ]

    survey_data.append(survey_entry)
    survey_id += 1

# Save into Excel
columns = [
    "SurveyID", "StudentID", "LecturerID", "ClassesID", "SubjectID",
    "Semester", "Year", "Overall_Satisfaction",
    "Usefulness", "Recommend_Course", "Clarity", "Fairness",
    "Use_of_Resources", "Most_Useful_Component", "Workload_Perception"
]

df = pd.DataFrame(survey_data, columns=columns)
file_name = "survey_results.xlsx"
df.to_excel(file_name, index=False)

print(f"Survey data saved to {file_name}")

# Close connection
cursor.close()
conn.close()


Survey data saved to survey_results.xlsx


In [28]:
'''
import random
import pyodbc
conn_str = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=DESKTOP-4TCLG8I;DATABASE=university;Trusted_Connection=yes'
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Query to get distinct subjects from 2025, Semester 1
cursor.execute("""
    SELECT DISTINCT SubjectID, MajorID, Semester, TeacherID, MethodOfConducting
    FROM SubjectInCertainSemester
    WHERE YearOfConduction = 2025 AND Semester = 1;
""")

# Fetch the results
data_2025 = cursor.fetchall()

# Fetch all teacher IDs to randomly assign if a teacher changes
cursor.execute("SELECT DISTINCT TeacherID FROM SubjectInCertainSemester;")
all_teachers = [row[0] for row in cursor.fetchall()]

# Prepare new data for insertion with YearOfConduction = 2026
data_2026 = []
for subject, major, semester, teacher, method in data_2025:
    if random.random() < 0.05:
     new_teacher = random.choice(all_teachers)
    else:
        new_teacher = teacher  # Keep the same teacher
    
    data_2026.append((2026, subject, major, semester, new_teacher, method))
random.shuffle(data_2026)

if data_2026:
    cursor.executemany("""
        INSERT INTO SubjectInCertainSemester (YearOfConduction, SubjectID, MajorID, Semester, TeacherID, MethodOfConducting)
        VALUES (?, ?, ?, ?, ?, ?);
    """, data_2026)

    conn.commit()
    print(f"Inserted {len(data_2026)} new records for 2026.")
else:
    print("No data found to insert.")

# Close the connection
conn.close()
'''

'\nimport random\nimport pyodbc\nconn_str = \'DRIVER={ODBC Driver 17 for SQL Server};SERVER=DESKTOP-4TCLG8I;DATABASE=university;Trusted_Connection=yes\'\nconn = pyodbc.connect(conn_str)\ncursor = conn.cursor()\n\n# Query to get distinct subjects from 2025, Semester 1\ncursor.execute("""\n    SELECT DISTINCT SubjectID, MajorID, Semester, TeacherID, MethodOfConducting\n    FROM SubjectInCertainSemester\n    WHERE YearOfConduction = 2025 AND Semester = 1;\n""")\n\n# Fetch the results\ndata_2025 = cursor.fetchall()\n\n# Fetch all teacher IDs to randomly assign if a teacher changes\ncursor.execute("SELECT DISTINCT TeacherID FROM SubjectInCertainSemester;")\nall_teachers = [row[0] for row in cursor.fetchall()]\n\n# Prepare new data for insertion with YearOfConduction = 2026\ndata_2026 = []\nfor subject, major, semester, teacher, method in data_2025:\n    if random.random() < 0.05:\n     new_teacher = random.choice(all_teachers)\n    else:\n        new_teacher = teacher  # Keep the same teach

In [29]:

conn.commit()
cursor.close()
conn.close()

ProgrammingError: Attempt to use a closed connection.